# Time Complexity Runner
Runs `mod_start_mass_simulator` across all (a, b) grid pairs in the configured range.
Each (a, b) pair gets N_SIMS simulations. Per-run step counts are parsed from stdout and saved to CSV.

**Run this notebook once to generate data. Do not re-run unless you want to regenerate.**

In [4]:
import subprocess
import re
import os
import multiprocessing as mp
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

print("CPUs available:", os.cpu_count())

CPUs available: 8


In [5]:
# --- Config ---
CPP_DIR   = Path("../cpp")
BINARY    = CPP_DIR / "mod_start_mass_simulator"
OUT_DIR   = Path("../output/time_complexity")
OUT_DIR.mkdir(parents=True, exist_ok=True)

A_MIN, A_MAX = 2, 50   # row range (inclusive)
B_MIN, B_MAX = 2, 50   # col range (inclusive)
N_SIMS       = 100     # simulations per (a, b) pair
MAX_WORKERS  = max(1, os.cpu_count() - 1)

In [6]:
# Regex to extract the last 5 fields: red, green, blue, steps, success
# Each simulation line ends with: ..., red, green, blue, steps, success
STATS_RE = re.compile(r'(\d+), (\d+), (\d+), (\d+), ([01])\s*$')

def run_and_parse(a: int, b: int, n: int, max_steps: int = 10000) -> list[dict]:
    """
    Runs mod_start_mass_simulator with log_verbose=True (no 6th arg).
    Parses stdout for per-run step counts.
    Returns list of dicts: {a, b, n_nodes, steps, success}
    """
    result = subprocess.run(
        [str(BINARY), str(a), str(b), str(n), str(max_steps)],
        cwd=str(CPP_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=True,
    )

    records = []
    for line in result.stdout.splitlines():
        m = STATS_RE.search(line)
        if m:
            records.append({
                "a":       a,
                "b":       b,
                "n_nodes": a * b,
                "steps":   int(m.group(4)),
                "success": int(m.group(5)),
            })
    return records

In [10]:
# Build list of all (a, b) pairs
pairs = [(a, b) for a in range(A_MIN, A_MAX + 1) for b in range(B_MIN, B_MAX + 1)]
print(f"{len(pairs)} grid configurations to run ({N_SIMS} sims each)")
print(f"Total simulations: {len(pairs) * N_SIMS:,}")
print(f"n_nodes range: {A_MIN*B_MIN} to {A_MAX*B_MAX}")

2401 grid configurations to run (100 sims each)
Total simulations: 240,100
n_nodes range: 4 to 2500


In [11]:
# --- Parallel execution ---
all_records = []
completed = 0

print(f"Running with {MAX_WORKERS} workers...")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(run_and_parse, a, b, N_SIMS): (a, b) for a, b in pairs}

    for fut in as_completed(futures):
        a, b = futures[fut]
        records = fut.result()
        all_records.extend(records)
        completed += 1
        if completed % 50 == 0 or completed == len(pairs):
            print(f"  {completed}/{len(pairs)} done  (records so far: {len(all_records):,})")

print(f"\nDone. Total records collected: {len(all_records):,}")

Running with 7 workers...
  50/2401 done  (records so far: 5,000)
  100/2401 done  (records so far: 10,000)
  150/2401 done  (records so far: 15,000)
  200/2401 done  (records so far: 20,000)
  250/2401 done  (records so far: 25,000)
  300/2401 done  (records so far: 30,000)
  350/2401 done  (records so far: 35,000)
  400/2401 done  (records so far: 40,000)
  450/2401 done  (records so far: 45,000)
  500/2401 done  (records so far: 50,000)
  550/2401 done  (records so far: 55,000)
  600/2401 done  (records so far: 60,000)
  650/2401 done  (records so far: 65,000)
  700/2401 done  (records so far: 70,000)
  750/2401 done  (records so far: 75,000)
  800/2401 done  (records so far: 80,000)
  850/2401 done  (records so far: 85,000)
  900/2401 done  (records so far: 90,000)
  950/2401 done  (records so far: 95,000)
  1000/2401 done  (records so far: 100,000)
  1050/2401 done  (records so far: 105,000)
  1100/2401 done  (records so far: 110,000)
  1150/2401 done  (records so far: 115,000)
  

In [12]:
# Save to CSV
df = pd.DataFrame(all_records)
csv_path = OUT_DIR / f"time_complexity_a{A_MIN}-{A_MAX}_b{B_MIN}-{B_MAX}_n{N_SIMS}.csv"
df.to_csv(csv_path, index=False)

print(f"Saved {len(df):,} rows to {csv_path}")
print(df.describe())

Saved 240,100 rows to ../output/time_complexity/time_complexity_a2-50_b2-50_n100.csv
                   a              b        n_nodes          steps   success
count  240100.000000  240100.000000  240100.000000  240100.000000  240100.0
mean       26.000000      26.000000     676.000000     501.432616       1.0
std        14.142165      14.142165     557.136691     425.654507       0.0
min         2.000000       2.000000       4.000000       4.000000       1.0
25%        14.000000      14.000000     216.000000     156.000000       1.0
50%        26.000000      26.000000     522.000000     386.000000       1.0
75%        38.000000      38.000000    1014.000000     739.000000       1.0
max        50.000000      50.000000    2500.000000    3041.000000       1.0


## Extended square-grid runs
Square grids only ($a = b$), extended to $a \in [3, 120]$ to push the curve-fit analysis to much larger $n$ values.

**Rough timing guide** (single pair, 100 sims, 1 worker):
| Grid | $n$ | Wall time |
|------|-----|-----------|
| 50×50 | 2 500 | ~17 s |
| 60×60 | 3 600 | ~40 s |
| 80×80 | 6 400 | ~2.2 min |
| 120×120 | 14 400 | ~10–12 min (est.) |

With `MAX_WORKERS` workers the large grids dominate. **Total estimated wall time: 30–90 min.** Reduce `SQ_MAX` or `SQ_SIMS` if you need faster results.

In [9]:
# --- Square-grid config ---
SQ_MIN       = 2      # smallest a (= b)
SQ_MAX       = 100     # largest a (= b)
SQ_SIMS      = 100    # simulations per grid
SQ_MAX_STEPS = 50000  # step limit per sim

sq_pairs = [(a, a) for a in range(SQ_MIN, SQ_MAX + 1)]
print(f"{len(sq_pairs)} square grids to run ({SQ_SIMS} sims each, max {SQ_MAX_STEPS} steps)")
print(f"Total simulations: {len(sq_pairs) * SQ_SIMS:,}")
print(f"n_nodes range: {SQ_MIN**2} to {SQ_MAX**2:,}")

99 square grids to run (100 sims each, max 50000 steps)
Total simulations: 9,900
n_nodes range: 4 to 10,000


In [ ]:
import time as _time

sq_records = []
sq_completed = 0
_t0 = _time.time()

print(f"Running {len(sq_pairs)} square grids with {MAX_WORKERS} workers...")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    sq_futures = {executor.submit(run_and_parse, a, b, SQ_SIMS, SQ_MAX_STEPS): (a, b) for a, b in sq_pairs}

    for fut in as_completed(sq_futures):
        a, b = sq_futures[fut]
        sq_records.extend(fut.result())
        sq_completed += 1
        elapsed = _time.time() - _t0
        rate = sq_completed / elapsed
        remaining = (len(sq_pairs) - sq_completed) / rate if rate > 0 else 0
        if sq_completed % 10 == 0 or sq_completed == len(sq_pairs):
            print(f"  {sq_completed}/{len(sq_pairs)}  "
                  f"elapsed {elapsed/60:.1f} min  "
                  f"est. remaining {remaining/60:.1f} min")

sq_df = pd.DataFrame(sq_records)
sq_csv = OUT_DIR / f"square_grids_a{SQ_MIN}-{SQ_MAX}_n{SQ_SIMS}.csv"
sq_df.to_csv(sq_csv, index=False)
print(f"\nSaved {len(sq_df):,} rows → {sq_csv}")
print(sq_df.describe())

Running 99 square grids with 7 workers...


  10/99  elapsed 0.0 min  est. remaining 0.0 min
  20/99  elapsed 0.0 min  est. remaining 0.1 min
  30/99  elapsed 0.1 min  est. remaining 0.2 min
  40/99  elapsed 0.4 min  est. remaining 0.6 min
  50/99  elapsed 1.2 min  est. remaining 1.2 min
  60/99  elapsed 3.1 min  est. remaining 2.0 min
  70/99  elapsed 7.3 min  est. remaining 3.0 min
  80/99  elapsed 15.2 min  est. remaining 3.6 min


## Large log-spaced square grids (O(n) vs O(n log n) discrimination)
Log-spaced square grids from ~n=10,000 to ~n=50,000. Even 2–5 sims per grid produces
a ~2–5σ separation between O(n) and O(n log n) at this scale.

**Timing guide per grid (10 sims, 1 worker):**
| Grid | n | ~time |
|------|---|-------|
| 100×100 | 10 000 | ~40 s |
| 141×141 | 19 881 | ~2.5 min |
| 200×200 | 40 000 | ~11 min |
| 224×224 | 50 176 | ~16 min |

With `MAX_WORKERS` workers all jobs run in parallel — wall time ≈ time of the single largest grid.

In [13]:
# --- Large log-spaced config ---
LG_SIMS      = 2       # ← adjust here (2–10 recommended; 2 = fast, 10 = smoother)
LG_MAX_STEPS = 100000  # safe headroom for n up to ~50k

# Log-spaced a values from 100 to 224 (n = 10k to ~50k)
import numpy as np
LG_SIZES = sorted(set(int(round(a)) for a in np.geomspace(100, 424, 12)))
lg_pairs = [(a, a) for a in LG_SIZES]

print(f"Grids: {LG_SIZES}")
print(f"n range: {LG_SIZES[0]**2:,} to {LG_SIZES[-1]**2:,}")
print(f"{len(lg_pairs)} grids × {LG_SIMS} sims = {len(lg_pairs)*LG_SIMS:,} total simulations")

Grids: [100, 114, 130, 148, 169, 193, 220, 251, 286, 326, 372, 424]
n range: 10,000 to 179,776
12 grids × 2 sims = 24 total simulations


In [14]:
import time as _time

lg_records = []
lg_completed = 0
_t0 = _time.time()

print(f"Running {len(lg_pairs)} large grids with {MAX_WORKERS} workers ({LG_SIMS} sims each)...")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    lg_futures = {executor.submit(run_and_parse, a, b, LG_SIMS, LG_MAX_STEPS): (a, b) for a, b in lg_pairs}

    for fut in as_completed(lg_futures):
        a, b = lg_futures[fut]
        lg_records.extend(fut.result())
        lg_completed += 1
        elapsed = _time.time() - _t0
        rate = lg_completed / elapsed
        remaining = (len(lg_pairs) - lg_completed) / rate if rate > 0 else 0
        print(f"  {lg_completed}/{len(lg_pairs)}  {a}×{b} (n={a*b:,})  "
              f"elapsed {elapsed/60:.1f} min  est. remaining {remaining/60:.1f} min")

lg_df = pd.DataFrame(lg_records)
lg_csv = OUT_DIR / f"large_logspaced_n{LG_SIMS}_{LG_SIZES[0]}_{LG_SIZES[-1]}sims.csv"
lg_df.to_csv(lg_csv, index=False)
print(f"\nSaved {len(lg_df):,} rows → {lg_csv}")
print(lg_df.groupby('n_nodes')['steps'].agg(['mean','std','count']))

Running 12 large grids with 7 workers (2 sims each)...


  1/12  100×100 (n=10,000)  elapsed 0.3 min  est. remaining 3.0 min
  2/12  114×114 (n=12,996)  elapsed 0.4 min  est. remaining 2.2 min
  3/12  130×130 (n=16,900)  elapsed 0.8 min  est. remaining 2.4 min
  4/12  148×148 (n=21,904)  elapsed 1.5 min  est. remaining 3.0 min
  5/12  169×169 (n=28,561)  elapsed 2.6 min  est. remaining 3.7 min
  6/12  193×193 (n=37,249)  elapsed 5.3 min  est. remaining 5.3 min
  7/12  220×220 (n=48,400)  elapsed 8.6 min  est. remaining 6.1 min
  8/12  251×251 (n=63,001)  elapsed 13.7 min  est. remaining 6.8 min
  9/12  286×286 (n=81,796)  elapsed 22.8 min  est. remaining 7.6 min
  10/12  326×326 (n=106,276)  elapsed 32.7 min  est. remaining 6.5 min
  11/12  372×372 (n=138,384)  elapsed 44.3 min  est. remaining 4.0 min
  12/12  424×424 (n=179,776)  elapsed 55.5 min  est. remaining 0.0 min

Saved 24 rows → ../output/time_complexity/large_logspaced_n2_100_424sims.csv
             mean          std  count
n_nodes                              
10000      9547.0  